# 02-fix. 클러스터 병합 파라미터 수정 및 재실행

**문제**: §5 병합에서 θ=0.8, τ=0.7이 한국어 짧은 라벨에 너무 관대하여 1,030 → 99 클러스터로 과도 병합 발생 (1개 클러스터에 94.8% 집중)

**수정**: θ=0.3, τ=0.9로 강화하여 재실행. §3(반복 클러스터링), §4(라벨링) 결과는 그대로 유지.

## 0. 환경 설정

In [1]:
# === 환경 설정 ===
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/5stone_experiment/1_clustering_test')

!pip install -q sentence-transformers scipy

import json
import unicodedata
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from scipy.special import iv as bessel_iv
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix
from sentence_transformers import SentenceTransformer
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print(f'작업 디렉토리: {os.getcwd()}')

Mounted at /content/drive
작업 디렉토리: /content/drive/MyDrive/5stone_experiment/1_clustering_test


## 0-1. 경로 및 파라미터 설정

In [2]:
# === 경로 설정 ===
BASE_DIR = Path('/content/drive/MyDrive/5stone_experiment/1_clustering_test')

PATH_CONFIG = {
    'processed': BASE_DIR / 'data' / 'processed',
    'checkpoints': BASE_DIR / 'checkpoints',
    'embeddings': BASE_DIR / 'embeddings',
    'clustering_ckpt_dir': BASE_DIR / 'checkpoints' / 'clustering',
    'pre_init_intents': BASE_DIR / 'data' / 'processed' / 'pre_init_intents.parquet',
    'initial_intentset': BASE_DIR / 'data' / 'processed' / 'initial_intentset.parquet',
    'intent_embeddings': BASE_DIR / 'embeddings' / 'intent_embeddings.npy',
}

# 수정된 병합 파라미터
MERGE_CONFIG = {
    'embed_model_name': 'snunlp/KR-SBERT-V40K-klueNLI-augSTS',
    'merge_geodesic_theta': 0.3,    # θ: 0.8 → 0.3 (대폭 강화)
    'merge_prob_tau': 0.9,           # τ: 0.7 → 0.9 (대폭 강화)
    'merge_kappa': 50.0,             # κ: 유지
}

print(f'수정된 파라미터:')
print(f'  θ (geodesic threshold): {MERGE_CONFIG["merge_geodesic_theta"]} (이전: 0.8)')
print(f'  τ (merge probability):  {MERGE_CONFIG["merge_prob_tau"]} (이전: 0.7)')

수정된 파라미터:
  θ (geodesic threshold): 0.3 (이전: 0.8)
  τ (merge probability):  0.9 (이전: 0.7)


## 1. 기존 라벨링 결과 로드
§4에서 생성한 `labels.json`을 그대로 사용한다.

In [3]:
# === 라벨링 결과 로드 ===

def load_labels() -> list:
    labels_path = PATH_CONFIG['clustering_ckpt_dir'] / 'labels.json'
    with open(labels_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    labeled_clusters = [(d['indices'], d['label']) for d in data]
    print(f'라벨링 결과 로드: {len(labeled_clusters)} 클러스터')
    print(f'  고유 라벨: {len(set(lbl for _, lbl in labeled_clusters))}')
    total_sents = sum(len(idx) for idx, _ in labeled_clusters)
    print(f'  총 문장: {total_sents:,}')
    return labeled_clusters


labeled_clusters = load_labels()

라벨링 결과 로드: 1030 클러스터
  고유 라벨: 926
  총 문장: 33,341


## 2. 기존 병합 체크포인트 삭제
이전 과도 병합 결과를 제거한다.

In [4]:
# === 기존 병합 체크포인트 삭제 ===

def remove_old_merge_checkpoint():
    old_path = PATH_CONFIG['clustering_ckpt_dir'] / 'merged_clusters.json'
    if old_path.exists():
        old_path.unlink()
        print(f'삭제 완료: {old_path}')
    else:
        print('기존 병합 체크포인트 없음')


remove_old_merge_checkpoint()

삭제 완료: /content/drive/MyDrive/5stone_experiment/1_clustering_test/checkpoints/clustering/merged_clusters.json


## 3. 라벨 임베딩 분포 사전 분석
병합 전에 라벨 임베딩 간 geodesic distance 분포를 확인하여 θ 값의 적절성을 검증한다.

In [5]:
# === 라벨 임베딩 분포 분석 ===

def analyze_label_distances(labeled_clusters: list) -> np.ndarray:
    """
    라벨 임베딩 간 geodesic distance 분포를 분석한다.
    """
    labels = [lbl for _, lbl in labeled_clusters]
    print(f'라벨 임베딩 생성 중... ({len(labels)}개)')
    model = SentenceTransformer(MERGE_CONFIG['embed_model_name'])
    label_embs = model.encode(labels, normalize_embeddings=True)

    # 모든 쌍의 geodesic distance 계산 (샘플링)
    K = len(label_embs)
    # 전체 계산 가능 (1030 × 1029 / 2 = ~530K pairs)
    cos_sim_matrix = label_embs @ label_embs.T
    np.fill_diagonal(cos_sim_matrix, 1.0)  # 자기 자신
    cos_sim_matrix = np.clip(cos_sim_matrix, -1.0, 1.0)
    geodesic_matrix = np.arccos(cos_sim_matrix)

    # 상삼각 행렬에서 거리 추출
    upper_tri = geodesic_matrix[np.triu_indices(K, k=1)]

    print(f'\n=== Geodesic distance 분포 ===')
    print(f'  총 쌍: {len(upper_tri):,}')
    for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]:
        count = (upper_tri < threshold).sum()
        pct = count / len(upper_tri) * 100
        print(f'  θ < {threshold}: {count:,} 쌍 ({pct:.2f}%)')

    print(f'\n  평균: {upper_tri.mean():.4f}')
    print(f'  중앙값: {np.median(upper_tri):.4f}')
    print(f'  최소: {upper_tri.min():.4f}')
    print(f'  최대: {upper_tri.max():.4f}')

    # θ=0.3에서 가장 가까운 쌍 샘플 확인
    print(f'\n=== θ < 0.3 근접 라벨 쌍 샘플 (상위 15) ===')
    close_pairs = []
    for i in range(K):
        for j in range(i + 1, K):
            if geodesic_matrix[i, j] < 0.3:
                close_pairs.append((i, j, geodesic_matrix[i, j]))
    close_pairs.sort(key=lambda x: x[2])
    for i, j, dist in close_pairs[:15]:
        print(f'  [{dist:.4f}] "{labels[i]}" ↔ "{labels[j]}"')

    return label_embs


label_embs = analyze_label_distances(labeled_clusters)

라벨 임베딩 생성 중... (1030개)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


=== Geodesic distance 분포 ===
  총 쌍: 529,935
  θ < 0.1: 144 쌍 (0.03%)
  θ < 0.2: 145 쌍 (0.03%)
  θ < 0.3: 168 쌍 (0.03%)
  θ < 0.4: 284 쌍 (0.05%)
  θ < 0.5: 715 쌍 (0.13%)
  θ < 0.6: 1,893 쌍 (0.36%)
  θ < 0.8: 15,437 쌍 (2.91%)
  θ < 1.0: 110,487 쌍 (20.85%)

  평균: 1.1187
  중앙값: 1.1299
  최소: 0.0000
  최대: 1.6224

=== θ < 0.3 근접 라벨 쌍 샘플 (상위 15) ===
  [0.0000] "문의-공항픽업서비스" ↔ "문의-공항픽업서비스"
  [0.0000] "문의-공항픽업서비스" ↔ "문의-공항픽업서비스"
  [0.0000] "문의-추가비용" ↔ "문의-추가비용"
  [0.0000] "문의-숙소예약및견적" ↔ "문의-숙소예약및견적"
  [0.0000] "문의-현금서비스" ↔ "문의-현금서비스"
  [0.0000] "문의-현금서비스" ↔ "문의-현금서비스"
  [0.0000] "확인요청-리볼빙결제" ↔ "확인요청-리볼빙결제"
  [0.0000] "문의-연회비" ↔ "문의-연회비"
  [0.0000] "문의-연회비" ↔ "문의-연회비"
  [0.0000] "문의-예약필요정보" ↔ "문의-예약필요정보"
  [0.0000] "신청-선택약정할인" ↔ "신청-선택약정할인"
  [0.0000] "요청-선결제" ↔ "요청-선결제"
  [0.0000] "확인요청-미납요금" ↔ "확인요청-미납요금"
  [0.0000] "확인요청-미납요금" ↔ "확인요청-미납요금"
  [0.0000] "문의-발리여행예약" ↔ "문의-발리여행예약"


## 4. 수정된 병합 실행
θ=0.3, τ=0.9로 강화된 파라미터로 병합을 재실행한다.

In [6]:
# === 병합 함수 ===

def compute_geodesic_distance(emb_a: np.ndarray, emb_b: np.ndarray) -> float:
    cos_sim = np.clip(np.dot(emb_a, emb_b), -1.0, 1.0)
    return np.arccos(cos_sim)


def vmf_log_prob(emb: np.ndarray, mu: np.ndarray, kappa: float, d: int) -> float:
    log_norm = (d / 2 - 1) * np.log(kappa) - (d / 2) * np.log(2 * np.pi) - np.log(bessel_iv(d / 2 - 1, kappa) + 1e-300)
    return log_norm + kappa * np.dot(emb, mu)


def compute_merge_probability(
    emb_i: np.ndarray,
    emb_j: np.ndarray,
    all_label_embs: np.ndarray,
    kappa: float,
) -> float:
    K = len(all_label_embs)
    d = emb_i.shape[0]
    pi_m = 1.0 / K
    total_prob = 0.0
    for mu_m in all_label_embs:
        log_p_i = vmf_log_prob(emb_i, mu_m, kappa, d)
        log_p_j = vmf_log_prob(emb_j, mu_m, kappa, d)
        total_prob += pi_m * np.exp(log_p_i + log_p_j)
    return total_prob


def merge_clusters_fixed(
    labeled_clusters: list,
    label_embs: np.ndarray,
) -> list:
    """
    수정된 파라미터로 클러스터 병합을 실행한다.
    """
    K = len(labeled_clusters)
    d = label_embs.shape[1]
    theta = MERGE_CONFIG['merge_geodesic_theta']
    tau = MERGE_CONFIG['merge_prob_tau']
    kappa = MERGE_CONFIG['merge_kappa']

    print(f'병합 실행: θ={theta}, τ={tau}, κ={kappa}')
    print(f'입력 클러스터: {K}')

    # Affinity graph 구축
    edges_i, edges_j = [], []
    edge_details = []  # 디버깅용

    for i in tqdm(range(K), desc='병합 후보 탐색'):
        for j in range(i + 1, K):
            dist = compute_geodesic_distance(label_embs[i], label_embs[j])
            if dist < theta:
                prob = compute_merge_probability(label_embs[i], label_embs[j], label_embs, kappa)
                if prob > tau:
                    edges_i.append(i)
                    edges_j.append(j)
                    edge_details.append({
                        'label_i': labeled_clusters[i][1],
                        'label_j': labeled_clusters[j][1],
                        'dist': round(dist, 4),
                        'prob': round(prob, 6),
                    })

    print(f'병합 엣지: {len(edges_i)}개')

    # 병합 엣지 상세 출력
    if edge_details:
        print(f'\n=== 병합 엣지 상세 ===')
        for ed in sorted(edge_details, key=lambda x: x['dist']):
            print(f'  [{ed["dist"]:.4f}, p={ed["prob"]:.6f}] "{ed["label_i"]}" ↔ "{ed["label_j"]}"')

    # Connected components
    if edges_i:
        data_ones = np.ones(len(edges_i) * 2)
        rows = edges_i + edges_j
        cols = edges_j + edges_i
        graph = csr_matrix((data_ones, (rows, cols)), shape=(K, K))
        n_components, component_labels = connected_components(graph, directed=False)
    else:
        n_components = K
        component_labels = np.arange(K)

    # 병합 실행
    merged = []
    for comp_id in range(n_components):
        member_cluster_ids = np.where(component_labels == comp_id)[0]
        merged_indices = []
        for cid in member_cluster_ids:
            merged_indices.extend(labeled_clusters[cid][0])
        biggest_cid = max(member_cluster_ids, key=lambda c: len(labeled_clusters[c][0]))
        merged_label = labeled_clusters[biggest_cid][1]
        merged.append((merged_indices, merged_label))

    print(f'\n병합 결과: {K} → {len(merged)} 클러스터 ({K - len(merged)}개 병합됨)')

    # 체크포인트 저장
    ckpt_path = PATH_CONFIG['clustering_ckpt_dir'] / 'merged_clusters.json'
    save_data = [{'indices': list(indices), 'label': label} for indices, label in merged]
    with open(ckpt_path, 'w', encoding='utf-8') as f:
        json.dump(save_data, f, ensure_ascii=False, indent=2)
    print(f'체크포인트 저장: {ckpt_path}')

    return merged


merged_clusters = merge_clusters_fixed(labeled_clusters, label_embs)

병합 실행: θ=0.3, τ=0.9, κ=50.0
입력 클러스터: 1030


병합 후보 탐색:   0%|          | 0/1030 [00:00<?, ?it/s]

병합 엣지: 168개

=== 병합 엣지 상세 ===
  [0.0000, p=inf] "문의-공항픽업서비스" ↔ "문의-공항픽업서비스"
  [0.0000, p=inf] "문의-공항픽업서비스" ↔ "문의-공항픽업서비스"
  [0.0000, p=inf] "문의-숙소예약및견적" ↔ "문의-숙소예약및견적"
  [0.0000, p=inf] "문의-현금서비스" ↔ "문의-현금서비스"
  [0.0000, p=inf] "문의-현금서비스" ↔ "문의-현금서비스"
  [0.0000, p=inf] "확인요청-리볼빙결제" ↔ "확인요청-리볼빙결제"
  [0.0000, p=inf] "문의-연회비" ↔ "문의-연회비"
  [0.0000, p=inf] "문의-연회비" ↔ "문의-연회비"
  [0.0000, p=inf] "문의-예약필요정보" ↔ "문의-예약필요정보"
  [0.0000, p=inf] "확인요청-미납요금" ↔ "확인요청-미납요금"
  [0.0000, p=inf] "문의-발리여행예약" ↔ "문의-발리여행예약"
  [0.0000, p=inf] "문의-요금제변경" ↔ "문의-요금제변경"
  [0.0000, p=inf] "문의-요금제변경" ↔ "문의-요금제변경"
  [0.0000, p=inf] "문의-요금제변경" ↔ "문의-요금제변경"
  [0.0000, p=inf] "문의-요금제변경" ↔ "문의-요금제변경"
  [0.0000, p=inf] "문의-요금제변경" ↔ "문의-요금제변경"
  [0.0000, p=inf] "문의-소액결제" ↔ "문의-소액결제"
  [0.0000, p=inf] "문의-연회비환불" ↔ "문의-연회비환불"
  [0.0000, p=inf] "확인요청-부가서비스" ↔ "확인요청-부가서비스"
  [0.0000, p=inf] "확인요청-카드사용가능여부" ↔ "확인요청-카드사용가능여부"
  [0.0000, p=inf] "요청-가상계좌발급" ↔ "요청-가상계좌발급"
  [0.0000, p=inf] "요청-가상계좌발급" ↔ "요청-가상계좌발급"
  [0.0000, p=inf

## 5. 결과 검증

In [7]:
# === 결과 검증 ===

def validate_merged(merged_clusters: list) -> None:
    n_clusters = len(merged_clusters)
    sizes = [len(c[0]) for c in merged_clusters]
    total_assigned = sum(sizes)

    print(f'=== 수정된 IntentSet 통계 ===')
    print(f'  클러스터 수: {n_clusters}')
    print(f'  할당된 문장: {total_assigned:,}')

    print(f'\n--- 클러스터 크기 분포 ---')
    print(f'  평균: {np.mean(sizes):.1f}, 중앙값: {np.median(sizes):.0f}')
    print(f'  최소: {np.min(sizes)}, 최대: {np.max(sizes)}')
    print(f'  1~5문장: {sum(1 for s in sizes if s <= 5)}')
    print(f'  6~20문장: {sum(1 for s in sizes if 6 <= s <= 20)}')
    print(f'  21~100문장: {sum(1 for s in sizes if 21 <= s <= 100)}')
    print(f'  100+문장: {sum(1 for s in sizes if s > 100)}')

    # 최대 클러스터 비중 확인 (과도 병합 검증)
    max_size = max(sizes)
    max_pct = max_size / total_assigned * 100
    print(f'\n  최대 클러스터 비중: {max_size:,}문장 ({max_pct:.1f}%)')
    if max_pct > 10:
        print(f'  ⚠ 최대 클러스터 비중이 10%를 초과합니다. 추가 파라미터 조정이 필요할 수 있습니다.')
    else:
        print(f'  ✓ 과도 병합 해소됨')

    # 고유 라벨
    unique_labels = set(lbl for _, lbl in merged_clusters)
    print(f'\n  고유 라벨: {len(unique_labels)}')

    # 행위 접두사 분포
    print(f'\n--- 행위 접두사 분포 ---')
    action_counts = Counter()
    for _, lbl in merged_clusters:
        action = lbl.split('-')[0] if '-' in lbl else lbl
        action_counts[action] += 1
    for action, cnt in action_counts.most_common(15):
        print(f'  {action}: {cnt}')

    # 상위 20 클러스터
    print(f'\n--- 상위 20 클러스터 (크기순) ---')
    sorted_c = sorted(merged_clusters, key=lambda c: len(c[0]), reverse=True)
    for indices, label in sorted_c[:20]:
        print(f'  [{len(indices):>4}문장] {label}')


validate_merged(merged_clusters)

=== 수정된 IntentSet 통계 ===
  클러스터 수: 907
  할당된 문장: 33,341

--- 클러스터 크기 분포 ---
  평균: 36.8, 중앙값: 30
  최소: 2, 최대: 417
  1~5문장: 186
  6~20문장: 164
  21~100문장: 514
  100+문장: 43

  최대 클러스터 비중: 417문장 (1.3%)
  ✓ 과도 병합 해소됨

  고유 라벨: 907

--- 행위 접두사 분포 ---
  문의: 546
  확인요청: 184
  요청: 108
  변경요청: 21
  정보제공: 8
  신청: 7
  해지요청: 7
  동의: 4
  거절: 4
  견적요청: 3
  요금납부: 3
  결제요청: 3
  추천요청: 2
  신고: 1
  납부요청: 1

--- 상위 20 클러스터 (크기순) ---
  [ 417문장] 문의-요금제변경
  [ 274문장] 문의-공항픽업서비스
  [ 242문장] 문의-결합할인혜택
  [ 225문장] 문의-예약필요정보
  [ 221문장] 문의-요금제
  [ 211문장] 요청-선결제
  [ 203문장] 확인요청-자동이체
  [ 201문장] 문의-결제방법
  [ 200문장] 문의-요금제변경가능여부
  [ 189문장] 문의-요금제변경및할인혜택
  [ 180문장] 요청-가상계좌발급
  [ 179문장] 문의-장기카드대출
  [ 172문장] 확인요청-결제내역
  [ 166문장] 요청-신용카드한도상향
  [ 160문장] 확인요청-미납요금
  [ 156문장] 확인요청-결제금액
  [ 147문장] 문의-카드한도상향
  [ 143문장] 확인요청-카드한도
  [ 140문장] 변경요청-카드결제일
  [ 140문장] 확인요청-카드사용가능여부


## 6. 최종 IntentSet 재저장
수정된 병합 결과로 `initial_intentset.parquet`와 `intentset_summary.parquet`를 덮어쓴다.

In [8]:
# === 최종 저장 ===

def save_fixed_intentset(merged_clusters: list) -> pd.DataFrame:
    intent_df = pd.read_parquet(PATH_CONFIG['pre_init_intents'])
    intent_df['intent_summary'] = intent_df['intent_summary'].apply(
        lambda x: unicodedata.normalize('NFC', x)
    )
    sentences = intent_df['intent_summary'].tolist()

    # 인덱스 매핑
    idx_to_cluster = {}
    idx_to_label = {}
    for cluster_id, (indices, label) in enumerate(merged_clusters):
        for idx in indices:
            idx_to_cluster[idx] = cluster_id
            idx_to_label[idx] = label

    result_df = intent_df.copy()
    result_df['cluster_id'] = result_df.index.map(lambda i: idx_to_cluster.get(i, -1))
    result_df['intent_label'] = result_df.index.map(lambda i: idx_to_label.get(i, 'UNASSIGNED'))

    assigned = result_df[result_df['cluster_id'] >= 0]
    unassigned = result_df[result_df['cluster_id'] < 0]
    print(f'할당됨: {len(assigned):,}, 미할당: {len(unassigned):,}')

    # 저장
    output_path = PATH_CONFIG['initial_intentset']
    result_df.to_parquet(output_path, index=False)
    print(f'저장: {output_path}')

    # 클러스터 요약
    cluster_summary = []
    for cluster_id, (indices, label) in enumerate(merged_clusters):
        cluster_summary.append({
            'cluster_id': cluster_id,
            'intent_label': label,
            'size': len(indices),
            'sample_sentences': json.dumps(
                [sentences[i] for i in indices[:5]],
                ensure_ascii=False,
            ),
        })

    summary_df = pd.DataFrame(cluster_summary)
    summary_path = PATH_CONFIG['processed'] / 'intentset_summary.parquet'
    summary_df.to_parquet(summary_path, index=False)
    print(f'클러스터 요약 저장: {summary_path}')

    return result_df


final_df = save_fixed_intentset(merged_clusters)
final_df.head()

할당됨: 33,341, 미할당: 684
저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed/initial_intentset.parquet
클러스터 요약 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed/intentset_summary.parquet


,source_id,source,consulting_category,split,intent_index,intent_summary,relevant_utterances,n_intents,is_fallback,raw_response,cluster_id,intent_label
0,90100,액티벤처,상품예약 및 결제,train,0,문의 — ▲▲월 ▲▲일 출발 5일 일정 2명 예약 가능 여부,"[""안녕하세요, ▲▲월 ▲▲일 출발로 5일 일정으로 2명 예약 가능한지 궁금해요.""...",8,False,,493,문의-숙소예약가능여부및가격
1,90100,액티벤처,상품예약 및 결제,train,1,문의 — 비치 좋은 리조트 조건 및 제외 지역,"[""비치가 좋은 곳으로, 쿠타 쪽으로 가는 택시가 힘들지 않았으면 좋겠어요."", ""...",8,False,,212,문의-리조트예약
2,90100,액티벤처,상품예약 및 결제,train,2,문의 — 희망 호텔 및 숙박 형태,"[""호텔은 그랜드 하얏트나 미라지를 생각하고 있어요."", ""자매 두 명이라 풀빌라를...",8,False,,41,문의-호텔예약가능여부
3,90100,액티벤처,상품예약 및 결제,train,3,요청 — 추천 호텔 문의,"[""추천해주실 만한 곳이 있으면 알려주세요.""]",8,False,,323,요청-숙소예약
4,90100,액티벤처,상품예약 및 결제,train,4,요청 — 담당자 직통 전화번호 문의,"[""담당자 직통 전화번호도 부탁드려요."", ""직통전화번호도 다시 한 번 알려주실 수...",8,False,,722,문의-상담원직통번호
